In [0]:
import csv
import sys
import re
import sqlite3 #reads nsys profiles exported to sqlite
import statistics
import subprocess #runs the harness / nsys from the MAKE DATA cells
from pathlib import Path
import matplotlib.pyplot as plot

#NOTE: the notebook runs from its own folder (YOLO/src/analysis/graphs), so the logs are 3 folders up in YOLO/benchmarking

BENCH_DIR = Path.cwd().parent.parent.parent / "benchmarking"
GRAPH_DIR = BENCH_DIR / "graphs" #location to save graphs to
HARNESS_DIR = Path.cwd().parent.parent / "substage_compiled" #harness main.py, used by the MAKE DATA cells
PLATFORM = "Jetson AGX Orin (CUDA)" #only used in graph titles

#adjust configuration of desired run
BATCH_SIZE = 1
STEP = None
OPTIMIZATION = None #e.g. "inductor"
PRECISION = None    #e.g. "bf16" or "fp32"

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
#LATENCY HELPER FUNCS

#read stage_timings.csv, one dict per row
#NOTE: rows are per image times in ms, columns:
#      timestamp,script,pass,device,batch_size,num_images,stage,substep,ms,step,optimization,precision

rows = list(csv.DictReader(open(BENCH_DIR / "stage_timings.csv")))
print(f"{len(rows)} rows in {BENCH_DIR / 'stage_timings.csv'}")
stages = {"pre_processing" : "#2a78d6", "inference" : "#eb6834", "post_processing" : "#1baf7a"} #dict to store each stage and their 
                                                                                                #associated color scheme

#helper func to find a run with desired configuration
def wanted_row(row):
    return ((BATCH_SIZE is None or row.get("batch_size") == str(BATCH_SIZE)) and (STEP is None or row.get("step") == str(STEP))
            and (OPTIMIZATION is None or row.get("optimization") == str(OPTIMIZATION)) and (PRECISION is None or row.get("precision") == PRECISION))

#helper func to find latest run
#NOTE: accepts pass_name as the type of harness run/pass associated with stored data point (e.g. substep_timing for substages,
#                                                                                           or wall_timing for stage wall times,
#                                                                                           or e2e for end to end batch times)
def latest_run(pass_name):
    matches = [r for r in rows if r["pass"] == pass_name and wanted_row(r)]
    if not matches:
        raise ValueError(f"NO {pass_name} ROWS MATCH THE SETTINGS (batch {BATCH_SIZE}, step {STEP}, opt {OPTIMIZATION}, precision {PRECISION})")
    latest = max(r["timestamp"] for r in rows) #finx most recent timestamp
    return [r for r in rows if r["timestamp"]==latest] #only grabs rows from most recent run

In [ ]:
#NOTE: CREATES A GRAPH THAT SHOWS ASYNC LATENCY BY SUBSTAGE (PER IMAGE) VIA CUDA EVENTS

#rows of most recent cuda events recorded in stage_timing.csv
run = [r for r in latest_run("substep_timing") if r["substep"] != "SUM_OF_STEPS"] #substeps only, in pipeline order
first = run[0] #grabs the first row taht matches cuda events
total = sum(float(r["ms"]) for r in run) #total time to find stage share percentages later on

for stage, color in stages.items():
    share = [r for r in run if r["stage"]==stage]
    bars = plot.barh([r["substep"] for r in share], [float(r["ms"]) for r in share], color = color, label = stage)
    plot.bar_label(bars, labels = [f"{float(r['ms']):.2f} ms ({float(r['ms']) / total * 100:.1f}%)" for r in share], padding = 3)


plot.xlim(0, max(float(r["ms"]) for r in run) * 1.35) #room for the labels past the longest bar
plot.xlabel("ms per image")
plot.title(f"{PLATFORM}: substep ASYNC latency per image (CUDA events)\n"
             f"batch {first['batch_size']}, step {first.get('step')}, {first.get('optimization')}, {first.get('precision')}, "
             f"{first['num_images']} images, {first['timestamp']}, total {total:.2f} ms/img")
plot.legend() #show what the colors are associated with from stages
plot.xlabel("ms per image") #add label
plot.gca().invert_yaxis() #make the earliest stages from preprocessing show up first top to bottom
plot.show()

In [ ]:
#HARDWARE HELPERS

#hardware graphs
POWER_RAIL = "VDD_GPU_SOC" #tegrastats rails: VDD_GPU_SOC, VDD_CPU_CV, VIN_SYS_5V0
TEMP_SENSOR = "gpu"        #tegrastats sensors: cpu, gpu, soc0, soc1, soc2, tj, ...
RUN = False #set True to make new data

if RUN:
    run([sys.executable, "main.py", "--setting", "hardware"] + harness_args())
    
#helper func that pulls one rail's watts and one sensor's degrees out of a tegrastats line
#e.g. "... VDD_GPU_SOC 5123mW/5000mW ... gpu@41.2C ..." (first mW number = right now)
def parse(line):
    watts = None
    degrees = None
    power = re.search(POWER_RAIL + r" (\d+)mW/", line) #e.g. finds "VDD_GPU_SOC 5123mW/"
    if power:
        watts = int(power.group(1)) / 1000 #mW to W
    temp = re.search(r"\b" + TEMP_SENSOR + r"@(-?[\d.]+)C", line) #e.g. finds "gpu@41.2C"
    if temp:
        degrees = float(temp.group(1))
    return watts, degrees

#every tegrastats reading: (time stamp, watts, degrees)
readings = []
for line in open(BENCH_DIR / "tegrastats.log"):    
    first_word = line.split(" ")[0] #harness writes time.time() at the start of every line
    try:
        t = float(line.split(" ", 1)[0])
    except ValueError: #line doesn't start with a time stamp
        continue
    watts, degrees = parse(line)
    readings.append((t,) + parse(line))
print(f"{len(readings)} tegrastats readings")

#helper func that is True when a hardware row matches the settings (no precision in hardware rows)
def wanted_hw(row):
    return ((BATCH_SIZE is None or row.get("batch_size") == str(BATCH_SIZE)) and (STEP is None or row.get("step") == str(STEP))
            and (OPTIMIZATION is None or row.get("optimization") == str(OPTIMIZATION)) and row.get("monitor") == "tegrastats")

#captures latest full pipeline window and latest window of every stage that match the settings
full = [r for r in csv.DictReader(open(BENCH_DIR / "hw_metrics.csv")) if wanted_hw(r)]
phases = [r for r in csv.DictReader(open(BENCH_DIR / "hw_stages.csv")) if wanted_hw(r)]
if not full or not phases:
    raise ValueError(f"NO HARDWARE RUN MATCHES THE SETTINGS (batch {BATCH_SIZE}, step {STEP}, opt {OPTIMIZATION})")

full = full[-1] #rows are added to the end of the csv, so the last one is the latest run
windows = {} #stage name -> (start, end) of its latest window
for row in phases:
    windows[row["stage"]] = (float(row["start"]), float(row["end"])) #later rows overwrite earlier ones
hw_stages = {"idle": "#8f8d87"}
hw_stages.update(stages)

#helper func that returns the watts and degrees read inside one window
def inside(start, end):
    watts = []
    degrees = []
    for t, w, d in readings:
        if start <= t <= end:
            if w is not None:
                watts.append(w)
            if d is not None:
                degrees.append(d)
    return watts, degrees

In [0]:

#NOTE: CREATE TWO CHARTS THAT SHOW RANGE OF POWER PER SUBSATAGE TESTED AND TEMPERATURE PER STAGE VS IDLE

names = [s for s in hw_stages if s in windows] #idle, pre, inference, post (the ones recorded)
                                               #list of stages to plot
plot.figure(figsize = (12, 4) ) #create a 12 x 4 figure that can fit two graphs 

#power per stage chart goes on left side of figure
plot.subplot(1, 2, 1)
for x, name in enumerate(names):
    watts = inside(*windows[name])[0] #every power reading inside this stage's window
    avg = sum(watts) / len(watts)
    below = max(avg - min(watts), 0) #length of the line below the dot, never negative (steady readings can round avg a hair past min/max)
    above = max(max(watts) - avg, 0) #length of the line above the dot
    plot.errorbar(x, avg, yerr = [[below], [above]], fmt = "o", color = hw_stages[name], capsize = 6, markersize = 8) #dot = avg, line = min to max
    plot.text(x + 0.08, avg, f"{avg:.1f} W", va = "center", fontsize = 9) #average next to the dot
    plot.text(x - 0.08, max(watts), f"max {max(watts):.1f}", ha = "right", va = "bottom", fontsize = 8, color = "#52514e") #max just above the top cap
    plot.text(x - 0.08, min(watts), f"min {min(watts):.1f}", ha = "right", va = "top", fontsize = 8, color = "#52514e") #min just below the bottom cap
plot.xticks(range(len(names)), names)
plot.xlim(-0.7, len(names) - 0.2) #room for the first min/max labels and the last avg label
plot.ylabel(f"{POWER_RAIL} (W)")
plot.title("power per stage", fontsize = 10)

#temp per stage chart goes on right side of figure
plot.subplot(1, 2, 2)
for x, name in enumerate(names):
    degrees = inside(*windows[name])[1] #every temperature reading inside this stage's window
    avg = sum(degrees) / len(degrees)
    below = max(avg - min(degrees), 0)
    above = max(max(degrees) - avg, 0)
    plot.errorbar(x, avg, yerr = [[below], [above]], fmt = "o", color = hw_stages[name], capsize = 6, markersize = 8)
    plot.text(x + 0.08, avg, f"{avg:.1f} C", va = "center", fontsize = 9)
    plot.text(x - 0.08, max(degrees), f"max {max(degrees):.1f}", ha = "right", va = "bottom", fontsize = 8, color = "#52514e")
    plot.text(x - 0.08, min(degrees), f"min {min(degrees):.1f}", ha = "right", va = "top", fontsize = 8, color = "#52514e")
plot.xticks(range(len(names)), names)
plot.xlim(-0.7, len(names) - 0.2)
plot.ylabel(f"{TEMP_SENSOR} (C)")
plot.title("temperature per stage", fontsize = 10)
    
plot.suptitle(f"{PLATFORM}: hardware range per stage\n"
              f"batch {full['batch_size']}, step {full['step']}, {full['optimization']}", fontsize = 11)
plot.tight_layout()
plot.savefig(GRAPH_DIR / "cuda_hw_range_per_stage.png", dpi = 200, bbox_inches = "tight")
plot.show()